# gfnx quickstart: environments, rewards, rollouts and Trajectory Balance

<a href="https://colab.research.google.com/github/d-tiapkin/gfnx/blob/main/notebooks/01_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook takes about five minutes on a CPU runtime. It covers:

1. the **decoupled** environment / reward API of `gfnx`,
2. vmapped rollouts with `gfnx.utils.forward_rollout`,
3. a complete **Trajectory Balance** (TB) training loop, compiled end to end with `jax.jit`,
4. evaluation against the exact target distribution.

Background on GFlowNets and the objectives is in the
[documentation](https://gfnx.readthedocs.io/en/latest/background/).

## Setup

`gfnx` supports Python 3.12–3.14. On Colab, the cell below installs it from GitHub. If
the install upgrades `numpy` or `jax` while an older version is already loaded, it
restarts the runtime once: then run the notebook again from the top (the install is
skipped the second time).

In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("gfnx") is None:
    if not (3, 12) <= sys.version_info[:2] <= (3, 14):
        raise RuntimeError(f"gfnx supports Python 3.12-3.14, this runtime has {sys.version}.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "gfnx @ git+https://github.com/d-tiapkin/gfnx.git",
            "equinox",
            "optax",
        ],
        check=True,
    )
    from importlib.metadata import version

    stale = [
        name
        for name in ("numpy", "jax")
        if name in sys.modules and sys.modules[name].__version__ != version(name)
    ]
    if stale and "google.colab" in sys.modules:
        print(f"Upgraded {stale}. Restarting the runtime: run the notebook again from the top.")
        os.kill(os.getpid(), 9)

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

import gfnx

print("jax", jax.__version__, "| devices:", jax.devices())

## 1. Environment and reward are separate objects

An environment only describes the **dynamics** of the generation DAG. Here, a
2-dimensional hypergrid of side 16: start at the origin, increment one coordinate per step,
and stop whenever you like. The reward is a separate module with its own parameters.

In [ ]:
DIM, SIDE = 2, 16

env = gfnx.HypergridEnvironment(dim=DIM, side=SIDE)
env_params = env.init(jax.random.PRNGKey(0))  # dynamics only

reward_module = gfnx.EasyHypergridRewardModule(side=SIDE)
reward_params = reward_module.init(jax.random.PRNGKey(0), env.reset())  # single dummy state

print("forward actions :", env.action_space.n, "(increment x, increment y, stop)")
print("backward actions:", env.backward_action_space.n)
print("max trajectory length:", env.max_steps_in_episode)

Every method acts on a **single** state, with no batch dimension:

In [ ]:
state = env.reset()
for action in [0, 0, 1, env.stop_action]:  # x += 1, x += 1, y += 1, stop
    obs, state, done, _ = env.step(state, jnp.int32(action), env_params)

print("terminal cell:", state.state, "| terminal:", bool(state.is_terminal))
print("log R(x) =", float(reward_module.log_reward(state, reward_params)))
print("valid backward actions:", env.get_backward_action_mask(state, env_params))

The hypergrid is small enough to enumerate, so the exact target distribution
$R(x)/Z$ is available. Its modes sit near the corners of the grid.

In [ ]:
true_dist = env.get_true_distribution(env_params, reward_module, reward_params)
log_Z = env.get_log_normalizing_constant(env_params, reward_module, reward_params)
print("log Z =", float(log_Z))

plt.imshow(true_dist.T, origin="lower", cmap="viridis")
plt.title("Target distribution R(x) / Z")
plt.colorbar()
plt.show()

## 2. Vmapped rollouts

`forward_rollout` samples one trajectory from a policy. A policy is any function
`(rng_key, obs, params) -> (logits, info)`; batching is just `jax.vmap`. Rewards are computed
afterwards, on the terminal states.

In [ ]:
def uniform_policy(rng_key, obs, params):
    logits = jnp.zeros(env.action_space.n)
    info = {"forward_logits": logits, "backward_logits": jnp.zeros(env.backward_action_space.n)}
    return logits, info


keys = jax.random.split(jax.random.PRNGKey(1), 1024)
traj_data, final_states, info = jax.vmap(
    lambda key: gfnx.utils.forward_rollout(key, uniform_policy, None, env, env_params)
)(keys)
log_rewards = jax.vmap(reward_module.log_reward, in_axes=(0, None))(final_states, reward_params)

print("trajectory tensor shape:", traj_data.action.shape, "= [batch, T + 1]")
print("mean trajectory length :", float(info["trajectory_length"].mean()))
print("mean log-reward        :", float(log_rewards.mean()))

## 3. Trajectory Balance

TB learns a forward policy $P_F$, here with a uniform backward policy $P_B$, and a scalar
$\log Z_\theta$ by minimising, for trajectories $\tau = (s_0, \ldots, s_n = x)$,

$$
\mathcal{L}_{\mathrm{TB}}(\tau) = \Big(\log Z_\theta + \sum_t \log P_F(s_{t+1} \mid s_t)
- \log R(x) - \sum_t \log P_B(s_t \mid s_{t+1})\Big)^2 .
$$

The policy is a small Equinox MLP that outputs forward logits. The loss re-runs the network
on the stored observations and reuses `gfnx.utils.forward_trajectory_log_probs`, which applies
the action masks and ignores padding steps.

In [ ]:
class Policy(eqx.Module):
    mlp: eqx.nn.MLP

    def __init__(self, key):
        self.mlp = eqx.nn.MLP(
            in_size=env.observation_space.shape[0],
            out_size=env.action_space.n,
            width_size=128,
            depth=2,
            key=key,
        )

    def __call__(self, obs):
        return {
            "forward_logits": self.mlp(obs),
            "backward_logits": jnp.zeros(env.backward_action_space.n),  # uniform P_B
        }


model = Policy(jax.random.PRNGKey(2))
model_params, model_static = eqx.partition(model, eqx.is_array)


def policy_fn(rng_key, obs, params):
    out = eqx.combine(params, model_static)(obs)
    return out["forward_logits"], out


def tb_loss(params, traj_data, log_rewards):
    net = eqx.combine(params["model"], model_static)
    outputs = jax.vmap(jax.vmap(net))(traj_data.obs)  # recompute logits: [B, T + 1, ...]
    traj_data = traj_data.replace(info=outputs)
    log_pf, log_pb = jax.vmap(
        lambda traj: gfnx.utils.forward_trajectory_log_probs(env, traj, env_params)
    )(traj_data)
    return jnp.mean((params["logZ"] + log_pf - log_rewards - log_pb) ** 2)


# logZ gets a larger learning rate than the network, as in the paper's baselines.
optimizer = optax.multi_transform(
    {"net": optax.adam(1e-3), "logZ": optax.adam(1e-1)},
    {"model": "net", "logZ": "logZ"},
)
params = {"model": model_params, "logZ": jnp.array(0.0)}
opt_state = optimizer.init(params)

One training step = sample a batch of trajectories with the current policy, score the
terminal states, take a gradient step. Early on, each step of the rollout picks a uniformly
random action with probability $\varepsilon$ (annealed from 1 to 0), which helps discover all
the modes. TB is off-policy, so exploration does not bias the loss. `jax.lax.scan` runs many
steps inside a single compiled program.

In [ ]:
BATCH_SIZE = 64
NUM_STEPS = 4_000
epsilon_schedule = optax.linear_schedule(1.0, 0.0, transition_steps=NUM_STEPS // 2)


def train_step(carry, inputs):
    params, opt_state = carry
    key, step = inputs
    epsilon = epsilon_schedule(step)

    def exploration_policy(rng_key, obs, model_params):
        logits, out = policy_fn(rng_key, obs, model_params)
        return gfnx.utils.apply_epsilon_greedy(rng_key, logits, epsilon), out

    keys = jax.random.split(key, BATCH_SIZE)
    traj_data, final_states, _ = jax.vmap(
        lambda k: gfnx.utils.forward_rollout(
            k, exploration_policy, params["model"], env, env_params
        )
    )(keys)
    log_rewards = jax.vmap(reward_module.log_reward, in_axes=(0, None))(
        final_states, reward_params
    )
    loss, grads = jax.value_and_grad(tb_loss)(params, traj_data, log_rewards)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return (params, opt_state), loss


@jax.jit
def train(params, opt_state, key, first_step, num_steps=500):
    steps = first_step + jnp.arange(num_steps)
    (params, opt_state), losses = jax.lax.scan(
        train_step, (params, opt_state), (jax.random.split(key, num_steps), steps)
    )
    return params, opt_state, losses

## 4. Evaluation against the exact target

`ExactDistributionMetricsModule` computes the *exact* terminal distribution of the current
policy by dynamic programming over the state graph, and reports its total variation (TV)
and KL distance to $R(x)/Z$.

In [ ]:
metric = gfnx.metrics.ExactDistributionMetricsModule(
    metrics=["tv", "kl"],
    env=env,
    fwd_policy_fn=policy_fn,
    reward_module=reward_module,
    batch_size=256,
)
metric_state = metric.init(
    jax.random.PRNGKey(3), metric.InitArgs(env_params=env_params, reward_params=reward_params)
)


@jax.jit
def evaluate(model_params, metric_state):
    metric_state = metric.process(
        metric_state,
        jax.random.PRNGKey(0),
        metric.ProcessArgs(policy_params=model_params, env_params=env_params),
    )
    return metric.get(metric_state)


print(
    "before training:", {k: float(v) for k, v in evaluate(params["model"], metric_state).items()}
)

In [ ]:
key = jax.random.PRNGKey(4)
history = []
for epoch in range(NUM_STEPS // 500):
    key, subkey = jax.random.split(key)
    params, opt_state, losses = train(params, opt_state, subkey, epoch * 500)
    scores = evaluate(params["model"], metric_state)
    history.append(float(scores["tv"]))
    print(
        f"step {(epoch + 1) * 500:5d} | loss {float(losses[-50:].mean()):.4f} | "
        f"logZ {float(params['logZ']):.3f} (true {float(log_Z):.3f}) | TV {history[-1]:.3f}"
    )

Samples from the trained policy should now cover all four modes in proportion to the
reward:

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(5), 20_000)
_, samples, _ = jax.vmap(
    lambda k: gfnx.utils.forward_rollout(k, policy_fn, params["model"], env, env_params)
)(keys)
empirical = env.get_empirical_distribution(samples, env_params)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, dist, title in [
    (axes[0], true_dist, "target R(x)/Z"),
    (axes[1], empirical, "GFlowNet samples"),
]:
    ax.imshow(dist.T, origin="lower", cmap="viridis")
    ax.set_title(title)
plt.show()

## Next steps

- [Notebook 2](https://colab.research.google.com/github/d-tiapkin/gfnx/blob/main/notebooks/02_vmap_seeds_and_configs.ipynb):
  train many seeds and hyperparameter settings in parallel with `jax.vmap`.
- [Notebook 3](https://colab.research.google.com/github/d-tiapkin/gfnx/blob/main/notebooks/03_custom_environment.ipynb):
  write your own environment and reward module.
- The [walkthrough](https://gfnx.readthedocs.io/en/latest/walkthrough/) dissects the full
  Detailed Balance baseline (`baselines/db_hypergrid.py`), with exploration, logging and
  checkpointing.